# 02 – Train dPL-SPARROW (full 20-year training)

Trains three ParamGenerator sub-networks jointly on all available years:

| Sub-network | Input features | Output used |
|---|---|---|
| `param_model` | catchment attributes (9) | N export rate α, delivery θ_D |
| `param_model_strm` | slope, mean discharge | stream attenuation θ_S |
| `param_model_res` | mean temperature | reservoir attenuation θ_R |

Checkpoints are saved every 10 epochs (starting at epoch 40) and whenever
validation loss improves.

**Prerequisites:** `01_data_preparation.ipynb`

**Outputs** (in `<output_dir>/`):
- `checkpoint_epoch_<n>.pth` – periodic checkpoints
- `checkpoint_last.pth` – final-epoch checkpoint
- `train_losses.json` – loss per epoch


In [ ]:
import os, random, json, warnings, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import SequentialLR, LinearLR, StepLR
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

from sparrow.model import SPARROW, ParamGenerator
from sparrow.utils import setup_seed, gpu_align

# ── User settings ─────────────────────────────────────────────────────────
data_dir     = '/your/path/to/data/'        # <-- CHANGE THIS
prepared_dir = './outputs/prepared/'
output_dir   = './outputs/train_full/'
os.makedirs(output_dir, exist_ok=True)

NUM_EPOCHS = 150
SEED       = 42
setup_seed(SEED)


In [ ]:
with open(os.path.join(prepared_dir, 'prepared_data.pkl'), 'rb') as f:
    prep = pickle.load(f)

input_data_huc12   = prep['input_data_huc12']
dlvdsgn_array      = prep['dlvdsgn_array']
source_columns     = prep['source_columns']
delivery_columns   = prep['delivery_columns']
stream_columns     = prep['stream_columns']
reservoir_columns  = prep['reservoir_columns']
input_columns      = prep['input_columns']
input_columns_strm = prep['input_columns_strm']
input_columns_res  = prep['input_columns_res']
scaler             = prep['scaler']
scaler_strm        = prep['scaler_strm']
scaler_res         = prep['scaler_res']
years              = prep['years']

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dlvdsgn = torch.tensor(dlvdsgn_array, dtype=torch.float32).to(device)
print(f"Device: {device}")


In [ ]:
# Re-build yearly DataLoaders on the current device
yearly_loaders = []
for year in years:
    yd = input_data_huc12[input_data_huc12['Year'] == year].copy()
    X_sc      = torch.tensor(scaler.transform(yd[input_columns].values),
                             dtype=torch.float32, device=device)
    X_sc_strm = torch.tensor(scaler_strm.transform(yd[input_columns_strm].values),
                             dtype=torch.float32, device=device)
    X_sc_res  = torch.tensor(scaler_res.transform(yd[input_columns_res].values),
                             dtype=torch.float32, device=device)
    obs = torch.tensor(yd['depvar'].values, dtype=torch.float32, device=device)
    obs[obs == 0] = float('nan')
    ds = TensorDataset(
        torch.tensor(yd[input_columns].values,       dtype=torch.float32, device=device),
        X_sc, X_sc_strm, X_sc_res,
        torch.tensor(yd[source_columns].values,    dtype=torch.float32, device=device),
        torch.tensor(yd[delivery_columns].values,  dtype=torch.float32, device=device),
        torch.tensor(yd[stream_columns].values,    dtype=torch.float32, device=device),
        torch.tensor(yd[reservoir_columns].values, dtype=torch.float32, device=device),
        torch.tensor(yd['rchtype'].values,    dtype=torch.int64,   device=device),
        torch.tensor(yd['hydseq'].values,     dtype=torch.float32, device=device),
        torch.tensor(yd['headflag'].values,   dtype=torch.int64,   device=device),
        torch.tensor(yd['from_node'].values,  dtype=torch.int64,   device=device),
        torch.tensor(yd['to_node'].values,    dtype=torch.int64,   device=device),
        torch.tensor(yd['frac'].values,       dtype=torch.float32, device=device),
        torch.tensor(yd['new_waterid'].values,dtype=torch.int64,   device=device),
        torch.tensor(yd['waterid'].values,    dtype=torch.int64,   device=device),
        torch.tensor(yd['iftran'].values,     dtype=torch.int64,   device=device),
        obs,
    )
    yearly_loaders.append(DataLoader(ds, batch_size=len(yd), shuffle=False))
train_loaders = yearly_loaders   # all years used for training


## Model initialisation

Three separate ParamGenerators share a single Adam optimiser so that
gradients flow through all sub-networks simultaneously.


In [ ]:
ns = len(source_columns)
nd = len(delivery_columns)

param_model = ParamGenerator(
    input_size=len(input_columns), hidden_size=32,
    num_source=ns, num_delivery=nd,
    num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns),
).to(device)

param_model_strm = ParamGenerator(
    input_size=len(input_columns_strm), hidden_size=8,
    num_source=ns, num_delivery=nd,
    num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns),
).to(device)

param_model_res = ParamGenerator(
    input_size=len(input_columns_res), hidden_size=8,
    num_source=ns, num_delivery=nd,
    num_stm_loss=len(stream_columns), num_res_loss=len(reservoir_columns),
).to(device)

sparrow_model = SPARROW().to(device)

optimizer = optim.Adam(
    list(param_model.parameters())
    + list(param_model_strm.parameters())
    + list(param_model_res.parameters()),
    lr=1e-3, weight_decay=1e-4,
)
criterion = nn.MSELoss()

scheduler = SequentialLR(
    optimizer,
    schedulers=[
        LinearLR(optimizer, start_factor=0.1, total_iters=10),
        StepLR(optimizer, step_size=50, gamma=0.5),
    ],
    milestones=[10],
)


## Training loop

In [ ]:
train_losses = []

for epoch in range(NUM_EPOCHS):
    param_model.train(); param_model_strm.train(); param_model_res.train()
    total_loss = 0.0

    shuffled = train_loaders.copy()
    random.shuffle(shuffled)

    for loader in shuffled:
        for batch in loader:
            (X_raw, X_sc, X_sc_strm, X_sc_res,
             S, Z_D, Z_S, Z_R,
             rtype, hseq, hflag, fnode, tnode, frac,
             cid, ocid, iftran, obs) = [b.to(device) for b in batch]

            coeffs      = param_model(X_sc)
            coeffs_strm = param_model_strm(X_sc_strm)
            coeffs_res  = param_model_res(X_sc_res)

            alpha   = coeffs[:, :ns]
            theta_D = coeffs[:, ns:ns + nd]
            theta_S = coeffs_strm[:, -2:-1]   # stream attenuation from stream sub-net
            theta_R = coeffs_res[:, -1:]       # reservoir attenuation from reservoir sub-net

            _, _, _, total_load, ocid_out = sparrow_model(
                S, Z_D, Z_S, Z_R, rtype, hseq, hflag, fnode, tnode,
                alpha, theta_D, theta_S, theta_R,
                frac, cid, ocid, dlvdsgn, iftran)

            total_load = total_load.to(device)
            pred, obs_a, mask = gpu_align(total_load, obs, ocid_out, ocid)

            if mask.sum() > 0 and (epoch + 1) != NUM_EPOCHS:
                loss = criterion(torch.log(pred[mask]), torch.log(obs_a[mask]))
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

    avg_loss = total_loss / len(train_loaders)
    train_losses.append(avg_loss)

    # Save checkpoint every 10 epochs starting at epoch 40
    if (epoch + 1) % 10 == 0 and (epoch + 1) >= 40:
        ckpt_path = os.path.join(output_dir, f'checkpoint_epoch_{epoch+1}.pth')
        torch.save({
            'epoch':            epoch + 1,
            'param_model':      param_model.state_dict(),
            'param_model_strm': param_model_strm.state_dict(),
            'param_model_res':  param_model_res.state_dict(),
            'optimizer':        optimizer.state_dict(),
            'scheduler':        scheduler.state_dict(),
            'train_loss':       train_losses,
        }, ckpt_path)
        print(f"Saved checkpoint: {ckpt_path}")

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1:4d} | LR: {lr:.6f} | Train Loss: {avg_loss:.6f}")

# Save final checkpoint
torch.save({
    'epoch':            NUM_EPOCHS,
    'param_model':      param_model.state_dict(),
    'param_model_strm': param_model_strm.state_dict(),
    'param_model_res':  param_model_res.state_dict(),
    'optimizer':        optimizer.state_dict(),
    'train_losses':     train_losses,
}, os.path.join(output_dir, 'checkpoint_last.pth'))

with open(os.path.join(output_dir, 'train_losses.json'), 'w') as f:
    json.dump({'train_losses': train_losses}, f)

print("\nTraining complete.")
